In [67]:
import os
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import KFold
from sklearn.model_selection import TimeSeriesSplit
import gc
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler, QuantileTransformer

In [68]:
PROCESSED_PATH = "../data/processed/features_static_after_split_TE.parquet"
df = pd.read_parquet(PROCESSED_PATH)
print(f"Loaded engineered dataset: {df.shape}")

Loaded engineered dataset: (590540, 327)


In [69]:
df.columns.tolist()

['isFraud',
 'TransactionDT',
 'TransactionAmt',
 'ProductCD',
 'card1',
 'card2',
 'card3',
 'card4',
 'card5',
 'card6',
 'addr1',
 'addr2',
 'dist1',
 'dist2',
 'P_emaildomain',
 'R_emaildomain',
 'C1',
 'C2',
 'C3',
 'C4',
 'C5',
 'C6',
 'C7',
 'C8',
 'C9',
 'C10',
 'C11',
 'C12',
 'C13',
 'C14',
 'D1',
 'D2',
 'D3',
 'D4',
 'D5',
 'D6',
 'D7',
 'D8',
 'D9',
 'D10',
 'D11',
 'D12',
 'D13',
 'D14',
 'D15',
 'M1',
 'M2',
 'M3',
 'M4',
 'M5',
 'M6',
 'M7',
 'M8',
 'M9',
 'V1',
 'V2',
 'V3',
 'V5',
 'V6',
 'V7',
 'V8',
 'V9',
 'V11',
 'V12',
 'V13',
 'V14',
 'V15',
 'V16',
 'V17',
 'V18',
 'V21',
 'V22',
 'V23',
 'V24',
 'V26',
 'V29',
 'V30',
 'V32',
 'V33',
 'V34',
 'V36',
 'V37',
 'V38',
 'V40',
 'V41',
 'V43',
 'V45',
 'V46',
 'V47',
 'V49',
 'V50',
 'V52',
 'V53',
 'V54',
 'V55',
 'V56',
 'V57',
 'V58',
 'V59',
 'V60',
 'V63',
 'V65',
 'V67',
 'V68',
 'V69',
 'V70',
 'V72',
 'V73',
 'V74',
 'V75',
 'V76',
 'V77',
 'V78',
 'V79',
 'V80',
 'V81',
 'V83',
 'V84',
 'V85',
 'V86',
 'V8

In [70]:
df['M9']

0         NaN
1         NaN
2           F
3         NaN
4         NaN
         ... 
590535      T
590536      F
590537    NaN
590538    NaN
590539      F
Name: M9, Length: 590540, dtype: category
Categories (2, str): ['F', 'T']

In [71]:
# split as in the part 3 LGBM
n_samples = len(df)
split_idx = int(n_samples * 0.8)

# These are tiny 1D integer arrays, not whole tables
idx_tr_time = np.arange(0, split_idx)
idx_val_time = np.arange(split_idx, n_samples)

Prepare data for NN - impute NaNs, scale features

In [91]:
TARGET='isFraud'
te_features = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M4']
drop_cols = [
    TARGET, 
    #'TransactionDT', # will need them for sequence building next step
    'Pseudo_UID',
    'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26' # sparse IP/proxy with >95% NaN
] + te_features  # Drops raw ProductCD, card4, etc.

features = [c for c in df.columns if c not in drop_cols]

cat_cols=['card1', 'card2', 'card3', 'card5',
                        'addr1', 'addr2',
                        'DeviceInfo',
                        'id_30', 'id_31', 'id_33', 'id_13', 'id_14', 'id_17', 'id_19', 'id_20', 'id_32', 'id_34'] 

bin_cols=['M1', 'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9', 
          'id_12', 'id_15', 'id_16', 'id_27', 'id_28', 'id_29', 'id_35', 'id_36', 'id_37', 'id_38', 
          'DeviceType']

block_bin_flags = [c for c in ['block_2_present', 'block_3_present', 'block_4_present'] if c in df.columns]

te_cols=[c for c in features if c.endswith('_target_enc')]

freq_cols = [c for c in df.columns if c.endswith('_freq')] # apply log1p before scaling

special_excluded = set(drop_cols + cat_cols + bin_cols + block_bin_flags + te_cols + freq_cols + ['uid_dt_diff'])
numeric_cols = [c for c in df.columns if c not in special_excluded]

In [94]:
def prepare_nn_fold_data(
    X_train_raw: pd.DataFrame, 
    X_val_raw: pd.DataFrame, 
    numeric_cols: list, 
    freq_cols: list,
    cat_cols: list, 
    bin_cols: list,
    block_bin_flags: list,
    te_cols: list,
    min_freq: int = 50,
    scaler_type: str = 'robust'
):
    """
    Fits imputation, log transforms, scaling, and categorical vocabularies
    strictly on the training fold, transforming both train and validation splits.
    
    Returns:
        X_tr_cont, X_va_cont : np.ndarray (float32) for Linear / BatchNorm layers
        X_tr_cat,  X_va_cat  : np.ndarray (int64) for nn.Embedding layers
        embedding_sizes      : list of (num_classes, emb_dim)
        cont_features        : list of continuous feature names
    """
    X_tr = X_train_raw.copy()
    X_va = X_val_raw.copy()

    # -------------------------------------------------------------------------
    # 1. Custom Feature: uid_dt_diff (First txn flag + log transform)
    # -------------------------------------------------------------------------
    if 'uid_dt_diff' in X_tr.columns:
        # Binary flag: 1.0 if this is the first transaction (was NaN), 0.0 otherwise
        X_tr['uid_dt_diff_is_first'] = X_tr['uid_dt_diff'].isna().astype(np.float32)
        X_va['uid_dt_diff_is_first'] = X_va['uid_dt_diff'].isna().astype(np.float32)

        # Convert Timedelta to total seconds if not already numeric
        for df_slice in [X_tr, X_va]:
            if pd.api.types.is_timedelta64_dtype(df_slice['uid_dt_diff']):
                df_slice['uid_dt_diff'] = df_slice['uid_dt_diff'].dt.total_seconds()

        # Impute NaN with 0.0, guard against negative offsets, then apply log1p
        tr_diff_clean = X_tr['uid_dt_diff'].fillna(0.0).astype(np.float64).clip(lower=0.0)
        va_diff_clean = X_va['uid_dt_diff'].fillna(0.0).astype(np.float64).clip(lower=0.0)

        X_tr['uid_dt_diff_log'] = np.log1p(tr_diff_clean).astype(np.float32)
        X_va['uid_dt_diff_log'] = np.log1p(va_diff_clean).astype(np.float32)

    # -------------------------------------------------------------------------
    # 2. Frequency Encodings: log1p Compression
    # -------------------------------------------------------------------------
    for col in freq_cols:
        if col in X_tr.columns:
            X_tr[col] = np.log1p(X_tr[col].fillna(0.0).clip(lower=0.0)).astype(np.float32)
            X_va[col] = np.log1p(X_va[col].fillna(0.0).clip(lower=0.0)).astype(np.float32)

    # -------------------------------------------------------------------------
    # 3. Tri-State & Binary Features (0.0 / 1.0 Flags)
    # -------------------------------------------------------------------------
    positive_values = {'T', 'True', True, 'Found', 1, '1.0', 1.0, 'mobile', 'found'}
    binary_feature_names = []

    # Clean known anomalous tokens
    if 'id_15' in X_tr.columns:
        X_tr['id_15'] = X_tr['id_15'].replace('Unknown', np.nan)
        X_va['id_15'] = X_va['id_15'].replace('Unknown', np.nan)

    for col in bin_cols:
        if col in X_tr.columns:
            pos_col = f'{col}_positive'
            nan_col = f'{col}_nan'

            X_tr[pos_col] = X_tr[col].isin(positive_values).astype(np.float32)
            X_tr[nan_col] = X_tr[col].isna().astype(np.float32)

            X_va[pos_col] = X_va[col].isin(positive_values).astype(np.float32)
            X_va[nan_col] = X_va[col].isna().astype(np.float32)

            binary_feature_names.extend([pos_col, nan_col])

    # Convert existing boolean flags to clean float32
    for col in block_bin_flags:
        if col in X_tr.columns:
            X_tr[col] = (X_tr[col] == True).astype(np.float32)
            X_va[col] = (X_va[col] == True).astype(np.float32)
            binary_feature_names.append(col)

    # Add the newly created first-transaction flag to unscaled binary list
    if 'uid_dt_diff_is_first' in X_tr.columns:
        binary_feature_names.append('uid_dt_diff_is_first')

    # -------------------------------------------------------------------------
    # 4. Continuous Missing Indicators & Imputation
    # -------------------------------------------------------------------------
    v_cols = [c for c in numeric_cols if c.startswith('V')]
    d_cols = [c for c in numeric_cols if c.startswith('D')]

    X_tr['total_null_count'] = X_tr[numeric_cols].isna().sum(axis=1).astype(np.float32)
    X_tr['v_null_count'] = X_tr[v_cols].isna().sum(axis=1).astype(np.float32)
    X_tr['d_null_count'] = X_tr[d_cols].isna().sum(axis=1).astype(np.float32)

    X_va['total_null_count'] = X_va[numeric_cols].isna().sum(axis=1).astype(np.float32)
    X_va['v_null_count'] = X_va[v_cols].isna().sum(axis=1).astype(np.float32)
    X_va['d_null_count'] = X_va[d_cols].isna().sum(axis=1).astype(np.float32)

    null_feature_names = ['total_null_count', 'v_null_count', 'd_null_count']

    # Impute numeric NaNs strictly with training medians
    train_medians = X_tr[numeric_cols].median().fillna(0.0)
    X_tr[numeric_cols] = X_tr[numeric_cols].fillna(train_medians)
    X_va[numeric_cols] = X_va[numeric_cols].fillna(train_medians)

    # Impute Target Encoding NaNs with training split mean
    if len(te_cols) > 0:
        te_fill = X_tr[te_cols].mean().fillna(0.035)
        X_tr[te_cols] = X_tr[te_cols].fillna(te_fill)
        X_va[te_cols] = X_va[te_cols].fillna(te_fill)

    # -------------------------------------------------------------------------
    # 5. Continuous Feature Scaling
    # -------------------------------------------------------------------------
    # All continuous features requiring distribution scaling
    cols_to_scale = list(numeric_cols) + list(freq_cols) + null_feature_names
    if 'uid_dt_diff_log' in X_tr.columns:
        cols_to_scale.append('uid_dt_diff_log')

    if scaler_type == 'quantile':
        scaler = QuantileTransformer(
            n_quantiles=1000, 
            output_distribution='normal', 
            random_state=42, 
            subsample=min(200_000, len(X_tr))
        )
    else:
        scaler = RobustScaler(with_centering=True, with_scaling=True)

    X_tr[cols_to_scale] = scaler.fit_transform(X_tr[cols_to_scale]).astype(np.float32)
    X_va[cols_to_scale] = scaler.transform(X_va[cols_to_scale]).astype(np.float32)

    # Sanitize any division-by-zero infinities
    X_tr[cols_to_scale] = X_tr[cols_to_scale].replace([np.inf, -np.inf], 0.0)
    X_va[cols_to_scale] = X_va[cols_to_scale].replace([np.inf, -np.inf], 0.0)

    # -------------------------------------------------------------------------
    # 6. Categorical Vocabulary Mapping for nn.Embedding
    # -------------------------------------------------------------------------
    embedding_sizes = []

    for col in cat_cols:
        tr_series = X_tr[col].astype('object').fillna("Missing").astype(str)
        va_series = X_va[col].astype('object').fillna("Missing").astype(str)

        # Identify frequent tokens strictly on the train slice
        top_cats = tr_series.value_counts()
        frequent_categories = set(top_cats[top_cats >= min_freq].index)
        frequent_categories.discard("Missing")

        # Map rare strings to 'Rare'
        tr_clean = tr_series.apply(lambda x: x if (x in frequent_categories or x == "Missing") else "Rare")
        va_clean = va_series.apply(lambda x: x if (x in frequent_categories or x == "Missing") else "Rare")

        # Build stable indexing dictionary
        unique_categories = ["Missing", "Rare"] + sorted(list(frequent_categories))
        cat_to_idx = {cat: idx for idx, cat in enumerate(unique_categories)}

        # Integer encode; unseen categories default to index 1 ('Rare')
        X_tr[col] = tr_clean.map(cat_to_idx).fillna(1).astype(np.int64)
        X_va[col] = va_clean.map(cat_to_idx).fillna(1).astype(np.int64)

        # Dimension rule: min(60, ceil(sqrt(cardinality)))
        num_classes = len(unique_categories)
        emb_dim = max(2, min(60, int(np.ceil(np.sqrt(num_classes)))))
        embedding_sizes.append((num_classes, emb_dim))

    # -------------------------------------------------------------------------
    # 7. Final Output Extraction
    # -------------------------------------------------------------------------
    cont_features = cols_to_scale + te_cols + binary_feature_names

    X_tr_cont = X_tr[cont_features].values.astype(np.float32)
    X_va_cont = X_va[cont_features].values.astype(np.float32)

    X_tr_cat = X_tr[cat_cols].values.astype(np.int64)
    X_va_cat = X_va[cat_cols].values.astype(np.int64)

    return X_tr_cont, X_va_cont, X_tr_cat, X_va_cat, embedding_sizes, cont_features

In [89]:
X_train_raw = df.iloc[idx_tr_time]
X_val_raw   = df.iloc[idx_val_time]
y_train     = df.iloc[idx_tr_time][TARGET].values.astype(np.float32)
y_val       = df.iloc[idx_val_time][TARGET].values.astype(np.float32)

In [95]:
# 2. Run preprocessing
X_tr_cont, X_va_cont, X_tr_cat, X_va_cat, emb_szs, cont_names = prepare_nn_fold_data(
    X_train_raw=X_train_raw,
    X_val_raw=X_val_raw,
    numeric_cols=numeric_cols,
    freq_cols=freq_cols,
    cat_cols=cat_cols,
    bin_cols=bin_cols,
    block_bin_flags=block_bin_flags,
    te_cols=te_cols,
    min_freq=50,
    scaler_type='robust'
)

In [97]:
# 3. Assert zero NaNs and verify shapes
assert not np.isnan(X_tr_cont).any(), "NaN found in X_tr_cont!"
assert not np.isnan(X_va_cont).any(), "NaN found in X_va_cont!"
assert not np.isnan(X_tr_cat).any(),  "NaN found in X_tr_cat!"
assert not np.isnan(X_va_cat).any(),  "NaN found in X_va_cat!"

print(f"X_tr_cont shape: {X_tr_cont.shape} (dtype: {X_tr_cont.dtype})")
print(f"X_tr_cat  shape: {X_tr_cat.shape}  (dtype: {X_tr_cat.dtype})")
print(f"Total categorical embedding tables: {len(emb_szs)}")

X_tr_cont shape: (472432, 318) (dtype: float32)
X_tr_cat  shape: (472432, 17)  (dtype: int64)
Total categorical embedding tables: 17


Part 2. Sequential track.  
Iterating row-by-row or using Python loops over ~590K transactions to construct sliding transaction histories will crash memory or take hours.To build sequences efficiently, we can use a NumPy index stride map:  
- Sort rows by (pseudo_UID, TransactionDT).  
- Compute each transaction's chronological rank within its user group using cumcount().  
- Map every transaction index backwards by up to $L$ steps within the same user boundary using pure array operations.  
- Gather sequential feature vectors into a 3D float32 tensor $\mathbf{X}_{\text{seq}} \in \mathbb{R}^{N \times L \times D_{\text{seq}}}$ and a boolean padding mask $\mathbf{M}_{\text{pad}} \in \{0, 1\}^{N \times L}$ (where True denotes padded/masked time steps for PyTorch's src_key_padding_mask).

Key Performance Benefits:Execution Time: Processes all ~590K transactions in ~1.5 to 2.5 seconds (compared to >30 minutes with .groupby().apply()).Memory Footprint: For $N=590{,}000$, $L=15$, and $D_{\text{seq}}=5$, X_seq consumes approximately 177 MB of RAM, fitting comfortably in memory.PyTorch Integration: seq_padding_mask matches PyTorch's native nn.TransformerEncoder(..., src_key_padding_mask=mask) directly, where True flags padded tokens to receive $-\infty$ attention weights.

In [105]:
import numpy as np
import pandas as pd
from typing import Tuple, List

def build_vectorized_user_sequences(
    df: pd.DataFrame,
    uid_col: str = 'pseudo_UID',
    time_col: str = 'TransactionDT',
    seq_feature_cols: List[str] = None,
    max_len: int = 15
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Constructs past transaction history tensors for each transaction in df
    without Python row-loops.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Source DataFrame containing all transactions.
    uid_col : str
        Column name for user/card proxy ID.
    time_col : str
        Column name used for temporal ordering.
    seq_feature_cols : List[str]
        List of continuous feature names to include in each sequence time step.
    max_len : int
        Maximum number of transactions to look back (including current). 
        Default 15 balances burst detection and Transformer compute.
        
    Returns:
    --------
    X_seq : np.ndarray, shape (N, max_len, D_seq), dtype float32
        3D tensor containing sequence features per transaction.
    padding_mask : np.ndarray, shape (N, max_len), dtype bool
        Boolean mask where True indicates padded (non-existent past) transactions,
        conforming directly to PyTorch's `src_key_padding_mask` convention.
    """
    if seq_feature_cols is None:
        raise ValueError("Must provide at least one feature column for seq_feature_cols.")
        
    n_samples = len(df)
    n_feats = len(seq_feature_cols)

    # 1. Preserve original row index so output matches the input DataFrame order
    df_work = df[[uid_col, time_col] + seq_feature_cols].copy()
    df_work['__orig_idx__'] = np.arange(n_samples)

    # 2. Sort by User and Time
    # Note: NaN UIDs are isolated and will not link sequences across unrelated transactions
    df_work = df_work.sort_values(by=[uid_col, time_col], kind='mergesort').reset_index(drop=True)

    # 3. Calculate sequence position within each user group
    # Transactions with NaN UID get isolated to individual length-1 sequences
    is_valid_uid = df_work[uid_col].notna().values # false if NaN
    
    # Fast cumulative count per UID
    uid_codes, _ = pd.factorize(df_work[uid_col].fillna("__NULL_UID__")) # get integer codes
    same_as_prev = (uid_codes[1:] == uid_codes[:-1]) & is_valid_uid[1:] & is_valid_uid[:-1]
    
    # Step-based cumcount using boundary detection
    boundary_marker = np.concatenate([[True], ~same_as_prev])  # a new group begins where same_as_prev is False
    group_starts = np.flatnonzero(boundary_marker) # indexes of non zero elements
    group_lengths = np.diff(np.concatenate([group_starts, [n_samples]])) # distance between successive start indices
    cum_counts = np.arange(n_samples) - np.repeat(group_starts, group_lengths) # resets the counter back to 0 at each boundary

    # 4. Build Backward-Looking Matrix of Indices (Shape: N x max_len)
    # lag 0 is current step, lag 1 is t-1, ..., lag (max_len - 1) is t-(max_len-1)
    current_sorted_indices = np.arange(n_samples)
    lags = np.arange(max_len - 1, -1, -1)  # Left-to-right chronological order: [t-(L-1), ..., t-1, t]

    # Shifted indices: current_index - lag
    candidate_indices = current_sorted_indices[:, None] - lags[None, :]

    # Determine valid past steps within the same user boundary:
    # A step is valid if: lag <= cum_counts (meaning we don't bleed into prior user)
    valid_mask = lags[None, :] <= cum_counts[:, None]

    # Map invalid/padded slots to index 0 (safe dummy row for array indexing)
    safe_indices = np.where(valid_mask, candidate_indices, 0)

    # 5. Extract Feature Matrix with a Dummy Zero Row
    # Raw feature matrix from sorted data
    raw_feats = df_work[seq_feature_cols].to_numpy(dtype=np.float32)
    # Replace any accidental NaN with 0.0 before tensor construction
    raw_feats = np.nan_to_num(raw_feats, nan=0.0, posinf=0.0, neginf=0.0)

    # Append dummy row at index 0 for all masked positions
    dummy_feats = np.vstack([np.zeros((1, n_feats), dtype=np.float32), raw_feats])
    # Offset safe_indices by +1 because row 0 is now the dummy zero row
    shifted_indices = np.where(valid_mask, safe_indices + 1, 0)

    # 6. Gather into 3D Tensor
    # Shape: (N, max_len, D_seq)
    X_seq_sorted = dummy_feats[shifted_indices]

    # 7. Invert valid_mask to PyTorch padding_mask (True = PAD / IGNORE)
    # Shape: (N, max_len)
    padding_mask_sorted = ~valid_mask

    # 8. Re-align arrays back to the original DataFrame row order
    orig_order = df_work['__orig_idx__'].to_numpy()
    
    X_seq = np.empty_like(X_seq_sorted)
    padding_mask = np.empty_like(padding_mask_sorted)

    X_seq[orig_order] = X_seq_sorted
    padding_mask[orig_order] = padding_mask_sorted

    return X_seq, padding_mask

In [107]:
# 1. Define 5-8 temporal features per sequence step
seq_features = [
    'TransactionAmt_log',       # np.log1p(df['TransactionAmt'])
    'uid_dt_diff_log',          # Elapsed seconds since last transaction
    'addr1_freq',               # Rarity of billing address
    'C1',                       # Phone/email verification frequency
    'C2'                        # Cardholder verification counter
]

df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'].clip(lower=0)).astype(np.float32)

# handle NaT first transactions for uid_dt_diff
MAX_ELAPSED_LOG = 365 * 24 * 3600
df['uid_dt_diff_log']=np.log1p(df['uid_dt_diff'].dt.total_seconds().fillna(MAX_ELAPSED_LOG).clip(lower=0))

# 2. Build 3D sequences (capped at last 15 transactions per user)
X_seq, seq_padding_mask = build_vectorized_user_sequences(
    df=df,
    uid_col='Pseudo_UID',
    time_col='TransactionDT',
    seq_feature_cols=seq_features,
    max_len=15
)

print(f"X_seq shape:        {X_seq.shape} ({X_seq.nbytes / (1024**2):.1f} MB)")
print(f"padding_mask shape: {seq_padding_mask.shape}")

X_seq shape:        (590540, 15, 5) (169.0 MB)
padding_mask shape: (590540, 15)


In [112]:
X_tr_seq = X_seq[idx_tr_time]
X_va_seq = X_seq[idx_val_time]

mask_tr  = seq_padding_mask[idx_tr_time]
mask_va  = seq_padding_mask[idx_val_time]

A standard Transformer expects a continuous hidden dimension ($d_{\text{model}}$), positional awareness, and attention masking. The sequence pipeline performs these operations:Linear Projection: Projects the raw $D_{\text{seq}}$ features (e.g., dimension 5) up to the model dimension $d_{\text{model}}$ (e.g., 64 or 128).Positional Encoding: Adds learned or sinusoidal positional vectors so the network understands the chronological order of transactions ($t-14, \dots, t-1, t$).Transformer Encoder: Self-attention layers compute pairwise correlations between past transactions, masked by src_key_padding_mask so padded dummy steps are assigned $-\infty$ attention weights.Sequence Pooling: Aggregates the $L$ step vectors into a single summary vector $\mathbf{h}_{\text{seq}} \in \mathbb{R}^{d_{\text{model}}}$ representing the user's recent behavioral trajectory. The most effective pooling strategy here is extracting the final non-padded step (the current transaction at index $t$), optionally concatenated with a masked mean over the past sequence.

In [111]:
import torch
import torch.nn as nn

class TransactionSequenceTransformer(nn.Module):
    def __init__(
        self,
        d_in: int = 5,           # Number of sequential features per step (D_seq)
        d_model: int = 64,       # Internal representation size
        nhead: int = 4,          # Attention heads (d_model must be divisible by nhead)
        num_layers: int = 2,     # Transformer encoder depth
        dim_feedforward: int = 128,
        dropout: float = 0.2,
        max_len: int = 15
    ):
        super().__init__()
        
        # 1. Project raw features into d_model dimensional space
        self.input_proj = nn.Linear(d_in, d_model)
        
        # 2. Learnable positional embeddings for chronological step positions
        self.pos_embedding = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)
        
        # 3. Transformer Encoder Stack
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True  # Inputs are (Batch, Seq_len, Dim)
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, 
            num_layers=num_layers
        )
        
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
        # 4. Dimension reduction head for fusion
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x_seq: torch.Tensor, padding_mask: torch.Tensor) -> torch.Tensor:
        """
        Parameters:
        -----------
        x_seq : torch.FloatTensor, shape (Batch, max_len, d_in)
        padding_mask : torch.BoolTensor, shape (Batch, max_len)
                       True indicates padded/dummy steps to mask out.
                       
        Returns:
        --------
        h_seq : torch.FloatTensor, shape (Batch, d_model)
        """
        B, L, _ = x_seq.shape
        
        # 1. Project input features
        h = self.input_proj(x_seq)  # (B, L, d_model)
        
        # 2. Add positional encoding
        h = h + self.pos_embedding[:, :L, :]
        h = self.dropout(h)
        
        # 3. Transformer Self-Attention
        # PyTorch requires src_key_padding_mask: True = ignore, False = attend
        # If an entire sequence is padded (edge case), unmask the last token to prevent NaN gradients
        all_padded = padding_mask.all(dim=-1)
        if all_padded.any():
            padding_mask = padding_mask.clone()
            padding_mask[all_padded, -1] = False
            
        h_trans = self.transformer_encoder(h, src_key_padding_mask=padding_mask)
        h_trans = self.layer_norm(h_trans)
        
        # 4. Pooling: Extract the representation of the current transaction (last token, index -1)
        # Because we aligned right-to-left, index -1 is always the target transaction t
        h_current = h_trans[:, -1, :]  # (B, d_model)
        
        # 5. Optional: Masked Mean Pooling across past context
        mask_expanded = (~padding_mask).unsqueeze(-1).float()  # 1 for valid, 0 for pad
        h_mean = (h_trans * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1.0)
        
        # Combine current state with aggregate sequence trajectory
        h_seq = self.out_proj(h_current + h_mean)
        
        return h_seq

Track 3 - device-text   
Device Info, id_30 and  id_31. - need cuda

In [116]:
df['device_text'] = (
    "device: " + df['DeviceInfo'].astype(str).replace("nan","missing") + 
    " os: " + df['id_30'].astype(str).replace("nan","missing") + 
    " browser: " + df['id_31'].astype(str).replace("nan","missing")
)

# extract only unique text strings and then map them back to the df
unique_texts=df['device_text'].unique().tolist()
print(f"Total rows: {len(df):,}, Unique device strings: {len(unique_texts):,}")

Total rows: 590,540, Unique device strings: 2,069


In [ ]:
from sentence_transformers import SentenceTransformer

# 'all-MiniLM-L6-v2' is fast, lightweight (~80MB), and outputs 384-dim vectors
text_encoder = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

unique_embeddings = text_encoder.encode(
    unique_texts, 
    batch_size=256, 
    show_progress_bar=True,
    convert_to_numpy=True
)

# 4. Map unique embeddings back to all rows via integer lookup
text_to_idx = {text: idx for idx, text in enumerate(unique_texts)}
row_indices = df['device_text'].map(text_to_idx).to_numpy()

# 5. Resulting matrix: shape (N, 384), dtype float32
X_text_emb = unique_embeddings[row_indices].astype(np.float32)

# 6. Slice to train and validation sets
X_tr_text = X_text_emb[idx_tr_time]
X_va_text = X_text_emb[idx_val_time]

c:\Users\ADMIN\Documents\DataScience\ieee-fraud-detection\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ADMIN\Documents\DataScience\ieee-fraud-detection\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Develop

Assemble the Multi-Modal PyTorch Dataset & DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class FraudMultiModalDataset(Dataset):
    def __init__(self, x_cont, x_cat, x_seq, seq_mask, x_text, y=None):
        self.x_cont = torch.tensor(x_cont, dtype=torch.float32)
        self.x_cat  = torch.tensor(x_cat, dtype=torch.int64)
        self.x_seq  = torch.tensor(x_seq, dtype=torch.float32)
        self.mask   = torch.tensor(seq_mask, dtype=torch.bool)
        self.x_text = torch.tensor(x_text, dtype=torch.float32)
        self.y      = torch.tensor(y, dtype=torch.float32) if y is not None else None

    def __len__(self):
        return len(self.x_cont)

    def __getitem__(self, idx):
        item = (self.x_cont[idx], self.x_cat[idx], self.x_seq[idx], self.mask[idx], self.x_text[idx])
        if self.y is not None:
            return (*item, self.y[idx])
        return item

train_loader = DataLoader(
    FraudMultiModalDataset(X_tr_cont, X_tr_cat, X_tr_seq, mask_tr, X_tr_text, y_train),
    batch_size=2048, shuffle=True, drop_last=True
)
val_loader = DataLoader(
    FraudMultiModalDataset(X_va_cont, X_va_cat, X_va_seq, mask_va, X_va_text, y_val),
    batch_size=4096, shuffle=False)

Join All 3 Branches in One Model Architecture

In [ ]:
class FraudMultiModalModel(nn.Module):
    def __init__(self, emb_szs, n_cont, d_seq, d_text=384):
        super().__init__()
        
        # 1. Static Branch (MLP)
        self.embeddings = nn.ModuleList([nn.Embedding(n, d) for n, d in emb_szs])
        total_emb_dim = sum(d for _, d in emb_szs)
        self.bn_cont = nn.BatchNorm1d(n_cont)
        self.static_mlp = nn.Sequential(
            nn.Linear(total_emb_dim + n_cont, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128)
        )
        
        # 2. Sequence Branch (Transformer)
        self.seq_transformer = TransactionSequenceTransformer(
            d_in=d_seq, d_model=64, nhead=4, num_layers=2
        )
        
        # 3. Device-Text Branch
        self.text_head = nn.Sequential(
            nn.Linear(d_text, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32)
        )
        
        # 4. Fusion Classifier Head
        # Concatenated sizes: 128 (static) + 64 (seq) + 32 (text) = 224
        self.classifier = nn.Sequential(
            nn.Linear(128 + 64 + 32, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)  # Output single raw logit
        )

    def forward(self, x_cont, x_cat, x_seq, seq_mask, x_text):
        # Branch 1: Static
        x_emb = torch.cat([emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)], dim=1)
        h_static = self.static_mlp(torch.cat([x_emb, self.bn_cont(x_cont)], dim=1))
        
        # Branch 2: Sequence
        h_seq = self.seq_transformer(x_seq, seq_mask)
        
        # Branch 3: Text
        h_text = self.text_head(x_text)
        
        # Fusion
        fused = torch.cat([h_static, h_seq, h_text], dim=1)
        return self.classifier(fused).squeeze(-1)